# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a full walkthrough for loading and exploring a colorectal cancer survivors dataset using the [`mlcroissant`](https://mlcroissant.mlcommons.org/) library in Python. All data entities (record sets, fields, and columns) are referenced by their unique `@id` fields following the Croissant schema standard.

### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure latest mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² colorectal cancer dataset. The Croissant `@id` for the dataset schema is used as the source.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Extract and display metadata using API attributes (not as dict)
print(f"Dataset: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, field `@id`s and explore schema structure. All identifiers refer to the Croissant entities.

Here, we print out all available record sets and their fields by `@id` for reference.

In [ ]:
# List all record sets in the dataset by @id and display their fields
print("Available record sets in this dataset:")

record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("[No record sets are explicitly declared in the top-level metadata.\n")
    print("However, mlcroissant may infer available record sets from files:")
    for record_set in dataset.available_record_sets:
        print(f"- Record set @id: {record_set}")
else:
    for rs in record_sets:
        print(f"- Record set @id: {rs['@id']}, name: {rs.get('name', None)}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    - Field @id: {field['@id']}, name: {field.get('name', None)}, dataType: {field.get('dataType', None)}")

# For this dataset, we demonstrate generic usage with default discovered record sets

print("\nExample record(s) from the first available record set:")
record_set_ids = list(dataset.available_record_sets)
if record_set_ids:
    sample_rs = record_set_ids[0]
    for idx, record in enumerate(dataset.records(record_set=sample_rs)):
        pprint.pprint(record)
        if idx >= 2:
            break

## 3. Data Extraction

Now, let's load all data from discovered record sets into Pandas dataframes for analysis. We will use each record set's `@id` as the key for referencing and extraction.

We print out the column `@id`s for further exploration.

In [ ]:
# Extract data from all available record sets
df_dict = {}
print("\nLoading dataframes for all available record sets by @id:")

for record_set_id in dataset.available_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    df_dict[record_set_id] = df
    print(f"- Record set @id: {record_set_id}, columns: {list(df.columns)} (num records: {df.shape[0]})")

# Use as example the first record set for downstream analysis
example_rs = list(df_dict.keys())[0]

print(f"\nFirst rows of DataFrame for record set @id: {example_rs}")
display(df_dict[example_rs].head(8))

## 4. Exploratory Data Analysis (EDA)

We'll perform data cleaning and simple analytics on the main table.

- **Filtering**: Let's select a numeric clinical field (e.g., age) for demonstration. We'll filter for age > 50 using its `@id` (update `<field_id>` with the correct one).
- **Normalization**: We'll normalize this numeric field.
- **Grouping**: We'll group by a categorical attribute (e.g., sex/gender or cancer location) via its `@id` if available.

You can use the printouts above to update the identifiers if more detail is needed.

In [ ]:
# Choose the main clinical record set and fields by @id
main_record_set_id = example_rs
main_df = df_dict[main_record_set_id]

# Inspect columns to select appropriate numeric and group fields
print("Available columns (fields by @id):")
print(main_df.columns.tolist())

# For this dataset, candidate numeric field: 'Age' (use exact @id used in this dataset)
# Let's try to automatically detect a field that can be 'Age' or similar
numeric_field_id = None
for col in main_df.columns:
    if 'Age' in col or 'age' in col:
        numeric_field_id = col
        break

if numeric_field_id and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    print(f"Using numeric field by @id: {numeric_field_id}")
    threshold = 50
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head(8))

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group/categorical field, e.g., for anatomical site, sex, or MSI status
    group_field_id = None
    for col in main_df.columns:
        if ('sex' in col.lower()) or ('gender' in col.lower()) or ('location' in col.lower()) or ('site' in col.lower()):
            group_field_id = col
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('Mean_Age')
        print(f"\nGrouped data (mean age) by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("Could not find a numeric 'Age' field by @id in this table for EDA.")
    print("Please review the fields and set 'numeric_field_id' to a valid numeric column @id.")

## 5. Visualization

Visualize the spread and distribution of the numeric field (e.g., age), and optionally, its relationship to a categorical variable (e.g., MSI status or anatomical site), referencing fields by their Croissant `@id`s. We'll use [Matplotlib](https://matplotlib.org/) or [Seaborn](https://seaborn.pydata.org/) for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric 'Age' field found for visualization. Edit code to set 'numeric_field_id' to a numeric column @id in the data.")

## 6. Conclusion
In this notebook, we explored the FAIR² colorectal cancer survivors dataset using `mlcroissant`, referencing all entities by their Croissant schema `@id`s for robust and portable workflows. We loaded metadata, performed EDA (including filtering, normalizing, and grouping clinical data), and visualized core distributions. This approach demonstrates best practices for FAIR and reproducible analysis on datasets described by Croissant schemas.